# Milestone 0 — Project Foundations
### Environment, config, logging, and clean shutdown — before any pipeline code

This notebook teaches the ideas behind `nova/config/loader.py`, `nova/logging_setup.py`, and `main.py`, which you already have as real files. We'll cover:

1. Why a project needs one central config file instead of scattered constants
2. YAML config loading, with caching, and why caching matters
3. Structured, per-stage logging — and why one logger per pipeline stage pays off later
4. Signal handling: how a long-running process shuts down *cleanly* instead of crashing
5. Dependency injection — the pattern that lets every later milestone's tests run without a microphone or GPU

> This notebook runs entirely inside this sandbox — no audio hardware or heavy model downloads are needed for anything in Milestone 0.


In [ ]:
# Nothing exotic needed for this one
import yaml
import logging
import sys
import time
import signal
print("Ready.")


## 1. Why one central config file?

Imagine `sample_rate = 16000` hardcoded in `capture.py`, `wake_word.py`, `vad.py`, and `transcriber.py` separately. Change it once and you now have four places to remember to update — and you *will* forget one, and NOVA will fail in a confusing way (e.g. VAD timestamps drifting because it assumes a different rate than STT).

The fix: **one `config.yaml`**, and every module reads from it via a single loader function. This is the same reasoning behind "single source of truth" in database design — one place owns the value, everything else references it.


In [ ]:
# A trimmed-down version of your real config.yaml, just for this notebook
sample_config_text = """
audio:
  sample_rate: 16000
  frame_ms: 30

wake_word:
  model_name: "hey_jarvis"
  threshold: 0.5

logging:
  level: "INFO"
  log_to_console: true
"""

config = yaml.safe_load(sample_config_text)
print(config)
print()
print("Every module just does: config['audio']['sample_rate']")
print("Change the number in ONE place, every module picks it up.")


## 2. Caching the config

Reading and parsing a YAML file is cheap once, but if `get_config()` re-read the file from disk on *every single call* — and remember, `wake_word.py`, `vad.py`, `transcriber.py`, `pipeline.py` all call it — you'd be hitting the filesystem constantly for a file that never changes while the app is running.

Your real `nova/config/loader.py` solves this with a **module-level cache** plus a **thread lock** (multiple pipeline stages may call `get_config()` from different threads — the audio callback thread and the main thread, for instance).


In [ ]:
import threading

_CONFIG_CACHE = None
_LOCK = threading.Lock()

def get_config_demo(loud=False):
    global _CONFIG_CACHE
    with _LOCK:
        if _CONFIG_CACHE is not None:
            if loud:
                print("(cache hit -- no file read)")
            return _CONFIG_CACHE
        if loud:
            print("(cache miss -- parsing YAML from disk)")
        _CONFIG_CACHE = yaml.safe_load(sample_config_text)
        return _CONFIG_CACHE

get_config_demo(loud=True)   # first call: parses
get_config_demo(loud=True)   # second call: cached
get_config_demo(loud=True)   # third call: still cached


Notice only the *first* call actually parses YAML. This is exactly what your real `get_config()` does — plus a `force_reload=True` escape hatch and a `path=` override, both used mainly in tests (see `reset_cache()` in `loader.py`, which the test suite calls between tests so they don't leak state into each other).


## 3. Structured, per-stage logging

A voice assistant has many moving parts running concurrently: audio capture, wake word, VAD, (later) the LLM brain, tool execution. If everything logs to one undifferentiated stream, debugging "why did NOVA respond slowly just now?" means reading every line by hand.

The fix: **namespace loggers by pipeline stage** — `nova.audio.capture`, `nova.wake_word`, `nova.stt`, etc. Then you can filter, or later (Milestone 8) measure latency, per stage.


In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-8s | %(name)-18s | %(message)s", datefmt="%H:%M:%S")

wake_word_log = logging.getLogger("nova.wake_word")
vad_log = logging.getLogger("nova.vad")
stt_log = logging.getLogger("nova.stt")

wake_word_log.info("Listening for 'hey nova'...")
vad_log.info("Speech started")
stt_log.info("Transcribed: 'what time is it'")
vad_log.info("Speech ended (1.4s captured)")


Notice the `name` column tells you instantly which stage produced each line, even interleaved. Your real `nova/logging_setup.py` wraps this pattern in `get_logger(stage_name)`, which just returns `logging.getLogger(f"nova.{stage_name}")` — simple, but it's the foundation Milestone 8's latency profiling depends on.

### Why `root.propagate = False` in the real code

Python's root logger, if left unconfigured, can end up printing a default "no handlers found" warning or double-printing lines if some imported library also configures logging. Setting `propagate = False` on the `nova` logger stops its messages from also bubbling up to Python's bare root logger and potentially being handled/printed twice.


## 4. Clean shutdown with signal handling

A voice assistant runs *forever* until you stop it — normally with Ctrl+C. The naive way to handle this is to let `KeyboardInterrupt` propagate and crash with a stack trace. That's bad UX and, worse, skips any cleanup (closing audio streams, flushing logs, releasing models).

The fix: **register a signal handler** that sets a flag instead of raising immediately, so your main loop can notice the flag, break out of its loop, and run cleanup code in a `finally` block.


In [ ]:
import signal
import time

shutdown_requested = False

def handle_shutdown(signum, frame):
    global shutdown_requested
    print(f"Received signal {signum} -- flagging shutdown (not crashing!)")
    shutdown_requested = True

# In real code this would be signal.SIGINT (Ctrl+C) and signal.SIGTERM (kill/service stop).
# We can't send ourselves a real OS signal cleanly inside a notebook cell, so simulate the callback directly:
handle_shutdown(signal.SIGINT, None)

print("Main loop would now see shutdown_requested=True and exit gracefully:")
if shutdown_requested:
    print("  -> logging 'Shutdown signal received. Cleaning up...'")
    print("  -> closing audio streams / releasing models (nothing to release yet at Milestone 0)")
    print("  -> logging 'NOVA stopped cleanly. Goodbye.'")


This is exactly the pattern in your real `main.py`: `signal.signal(signal.SIGINT, _handle_shutdown_signal)` registers the handler, the `while not _shutdown_requested: time.sleep(0.2)` loop polls the flag, and a `finally:` block guarantees the shutdown-logging code runs even if something inside the loop raised an exception.


## 5. Dependency injection — the pattern behind every later test

Here's the idea that makes Milestone 1 and 2's tests possible **without a microphone or downloaded model weights**: instead of a class always creating its own dependency internally, let the caller optionally *hand it one*.


In [ ]:
class RealSensor:
    def read(self):
        # imagine this talks to real hardware
        return "real hardware reading"

class Thermostat:
    def __init__(self, sensor=None):
        # If nobody hands us a sensor, create the real one.
        # If they do (e.g. a test), use theirs instead.
        self._sensor = sensor or RealSensor()

    def check_temperature(self):
        return self._sensor.read()

# Production code: uses the real sensor automatically
prod_thermostat = Thermostat()
print(prod_thermostat.check_temperature())

# Test code: injects a fake, no real hardware touched
class FakeSensor:
    def read(self):
        return "72 degrees (fake)"

test_thermostat = Thermostat(sensor=FakeSensor())
print(test_thermostat.check_temperature())


This is precisely why `WakeWordDetector.__init__(self, config=None, model=None)` accepts a `model` argument, why `SpeechSegmenter.__init__(self, is_speech_fn, config=None)` takes a function instead of loading Silero itself, and why `Transcriber.__init__(self, config=None, model=None)` follows the same shape. Every one of Milestone 1 and 2's tests passes a fake in through these seams — nothing about the *logic being tested* (does the state machine transition correctly? does text get joined correctly?) requires real hardware or real model weights.

### Exercise
Open `tests/test_milestone0_bootstrap.py` and `tests/test_milestone1_audio.py` side by side. For each test, identify: what's being injected, and what real behavior is it standing in for?


## 6. Putting it together: what `main.py` actually does at boot

```
1. get_config()          -- load + cache config.yaml
2. setup_logging()       -- configure console + file handlers, once
3. register signal handlers for SIGINT/SIGTERM
4. log "Booting NOVA v0.0.1"
5. log "NOVA ready."
6. loop: sleep 0.2s, check shutdown flag
7. on Ctrl+C: log "Shutdown signal received...", clean up, log "...Goodbye.", exit 0
```

Nothing here touches audio yet -- that's the entire point of Milestone 0: prove the skeleton (config, logging, lifecycle) is solid *before* adding anything that can fail in more interesting ways (a missing microphone, a model that won't load, etc).

**Next notebook:** Milestone 1 — Audio Input Pipeline (you already have `01_audio_pipeline_foundations.ipynb` for this one).
